# **Simulation Tool 2026**

## Install dependencies

In [ ]:
# %pip install rocketpy==1.12.1
# %pip install CoolProp
# %pip install openmeteo-requests requests-cache retry-requests pandas plotly colorama
# %pip install --upgrade nbformat
# %pip install "niquests==3.18.8" "urllib3-future==2.20.904"
# %pip install nbstripout
# %pip install nbconvert
# %pip install pyproj pymap3d
# %pip install pydantic
# %pip install tabulate

## Imports

In [ ]:
import simulation.simulation as sim
import simulation.outputs as outputs
import simulation.utils as utils
import simulation.reanalysis as reanalysis
from pathlib import Path

In [ ]:
# reload imported Python modules when you change their .py files
%load_ext autoreload
%autoreload 2

## Configuration
The length unit chosen here is millimeters to keep the values more readable. If necessary, values should be converted accordingly.

In [ ]:
PROJECT = "ALBATROSS"
CONFIG_PATH = Path(f"{PROJECT}/config.json")
ZONES_PATH = Path(f"{PROJECT}/zones.json")

# TODO: config generation

# True  -> use ONLY the standard atmosphere
# False -> use every envType from config EXCEPT the standard atmosphere (Windy / custom / etc.)
USE_ONLY_STANDARD_ENVIRONMENT = False

params = utils.load_config(CONFIG_PATH)

env_types = utils.ensure_list(params.config.environment.envType)
if USE_ONLY_STANDARD_ENVIRONMENT:
    env_types = ["standard_atmosphere"]
else:
    env_types = [e for e in env_types if e != "standard_atmosphere"]
params.config = params.config.model_copy(update={
    "environment": params.config.environment.model_copy(update={"envType": env_types})
})

print("Config loaded")
print("Active envTypes:", params.config.environment.envType)

## Zones

In [ ]:
exclusion_zones, buffer_zones, exclusion_zone_safety_margin = utils.load_zones(ZONES_PATH)
# Add buffer zones around exclusion zones
buffer_zones.update(utils.scale_zones(exclusion_zones, exclusion_zone_safety_margin))

# buffer_zones = {}                # remove all buffer zones; check only hard exclusions
# buffer_zones = exclusion_zones   # use exclusions as buffers (no margin, discard buffer zones from zones.json)

outputs.plot_landing_positions_with_modes(params, exclusion_zones, buffer_zones, plot_name="Zones", zones_only=True)

## Environments Initialization
In this section the environments are initialized.

In [ ]:
sim.create_environment(params)

## Simulation
### Tanks / Engine

In [ ]:
sim.create_engine(params)

### Rocket
Create Nosecone, Tailcone, Fins, Parachutes.

RocketPy definitions:
- dry mass = rocket with motor but without propellant
- Rocket Loaded Mass = Wet mass
- Rocket Center of Dry Mass - Nozzle Exit = Rocket Center of Dry Mass from bottom

In [ ]:
sim.create_rocket(params)

## Flight

In [ ]:
sim.create_flight(params)

## Output

A heading direction is only safe if <u>every</u> simulated rocket scenario (nominal/no main/ballistic/payload), every inclination and every environment stays outside the buffer zones.

In [ ]:
reanalysis.build_reanalysis_artifacts(params)

outputs.run_notebook_display_mode(params, exclusion_zones, buffer_zones)

reanalysis.run_reanalysis_comparison(params)

if params.export_kml:
    exported_kml_files = outputs.export_all_kml(params)
    # exported_csv_files = outputs.export_all_trajectory_csv(params)

outputs.export_notebook_to_html("simulation_orchestration.ipynb", params.project_path / "report.html")